In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm
import json
from typing import Optional, Tuple
import os

# Set device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


In [2]:
TENSOR_DIR = 'tensors_gpt/'  # Update to your path

# Load metadata
with open('tensors_rep/metadata.json', 'r') as f:
    metadata = json.load(f)

# Load tensors
X_train = np.load(TENSOR_DIR + 'X_train.npy')
X_val = np.load(TENSOR_DIR + 'X_val.npy')
X_test = np.load(TENSOR_DIR + 'X_test.npy')
y_train = np.load(TENSOR_DIR + 'y_train.npy')
y_val = np.load(TENSOR_DIR + 'y_val.npy')
y_test = np.load(TENSOR_DIR + 'y_test.npy')
mask_train = np.load(TENSOR_DIR + 'mask_train.npy')
mask_val = np.load(TENSOR_DIR + 'mask_val.npy')
mask_test = np.load(TENSOR_DIR + 'mask_test.npy')

# Get feature lists
original_features = metadata['original_features']   # 15 features
gap_features = metadata['gap_features']             # 15 features
all_features = metadata['all_features']             # 30 features
T = metadata['T']

print(f"X_train: {X_train.shape}  (N, T, F)")
print(f"mask_train: {mask_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"Train mortality: {y_train.mean()*100:.1f}%")
print(f"Features: {len(all_features)} total ({len(original_features)} original + {len(gap_features)} time gaps)")

X_train: (45756, 48, 30)  (N, T, F)
mask_train: (45756, 48)
y_train: (45756,)
Train mortality: 11.0%
Features: 30 total (15 original + 15 time gaps)


In [7]:
def remove_zero_mask_samples(X, mask, y):
    valid = mask.sum(axis=1) > 0
    print(f"Removing {(~valid).sum()} samples with zero valid hours")
    return X[valid], mask[valid], y[valid]

X_train, mask_train, y_train = remove_zero_mask_samples(X_train, mask_train, y_train)
X_val, mask_val, y_val = remove_zero_mask_samples(X_val, mask_val, y_val)
X_test, mask_test, y_test = remove_zero_mask_samples(X_test, mask_test, y_test)

Removing 0 samples with zero valid hours
Removing 0 samples with zero valid hours
Removing 0 samples with zero valid hours


In [3]:
class ICUDataset(Dataset):
    def __init__(self, X, mask, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.mask = torch.tensor(mask.astype(bool), dtype=torch.bool)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.mask[idx], self.y[idx]

In [5]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        targets = targets.float()

        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = probs * targets + (1 - probs) * (1 - targets)

        loss = self.alpha * (1 - pt) ** self.gamma * bce
        return loss.mean()

In [22]:
criterion = FocalLoss(alpha=0.5, gamma=2.0)

In [4]:
# no TSMOTE — use original training data as is
print(f"Train samples: {len(y_train)}, mortality: {y_train.mean()*100:.1f}%")

# skip directly to dataset creation
train_ds  = ICUDataset(X_train, mask_train, y_train)
val_ds    = ICUDataset(X_val,   mask_val,   y_val)
test_ds   = ICUDataset(X_test,  mask_test,  y_test)

BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# weighted loss instead of TSMOTE
n_neg      = (y_train == 0).sum()
n_pos      = (y_train == 1).sum()
pos_weight = torch.tensor([(n_neg / n_pos) * 0.8], dtype=torch.float32).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"pos_weight: {pos_weight.item():.2f}")

Train samples: 45756, mortality: 11.0%
pos_weight: 6.49


In [15]:
from model import build_training_components

model, criterion, optimizer = build_training_components(X_train, y_train, DEVICE,d_model = 64, n_heads = 4, n_layers=4, lstm_hidden = 128)

In [16]:
N_FEATURES = X_train.shape[2]
print(f"Model input features: {N_FEATURES}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Model input features: 20
Total parameters: 617,922


In [9]:
def train_one_epoch(model, loader, criterion, optimizer, device, grad_clip=1.0):
    model.train()
    total_loss = 0.0

    for xb, mb, yb in loader:
        xb = xb.to(device)
        mb = mb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(xb, mask=mb, return_attentions=False)
        loss = criterion(logits, yb)
        loss.backward()

        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)

In [9]:
def find_best_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> Tuple[float, float]:
    from sklearn.metrics import f1_score

    best_t = 0.5
    best_f1 = -1.0
    for t in np.linspace(0.4, 0.9, 101):
        y_pred = (y_prob >= t).astype(np.int64)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    return best_t, best_f1

In [22]:
import numpy as np
import torch
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    roc_auc_score
)


@torch.no_grad()
def evaluate_model(model, loader, device, verbose):
    model.eval()

    all_probs = []
    all_labels = []

    for xb, mb, yb in loader:
        xb = xb.to(device)
        mb = mb.to(device)
        yb = yb.to(device)

        logits, _ = model(xb, mask=mb)

        probs = torch.sigmoid(logits)   # ✅ IMPORTANT

        all_probs.append(probs.cpu().numpy())
        all_labels.append(yb.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    # -------------------------------
    # 🔍 Find best threshold
    # -------------------------------
    best_f1 = 0
    best_t = 0.5

    for t in np.linspace(0.5, 0.8, 50):
        preds = (all_probs >= t).astype(int)
        f1 = f1_score(all_labels, preds, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    # Final predictions
    final_preds = (all_probs >= best_t).astype(int)

    # -------------------------------
    # 📊 Metrics
    # -------------------------------
    acc = accuracy_score(all_labels, final_preds)
    precision = precision_score(all_labels, final_preds, zero_division=0)
    recall = recall_score(all_labels, final_preds, zero_division=0)
    f1 = f1_score(all_labels, final_preds, zero_division=0)
    auc = roc_auc_score(all_labels, all_probs)

    # -------------------------------
    # 📉 Confusion Matrix
    # -------------------------------
    cm = confusion_matrix(all_labels, final_preds)

    if verbose:
        print("\n===== EVALUATION =====")
        print(f"Best Threshold: {best_t:.3f}")
        print(f"Accuracy:  {acc:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1 Score:  {f1:.4f}")
        print(f"AUC:       {auc:.4f}")

        print("\nConfusion Matrix:")
        print(cm)

        print("\nClassification Report:")
        print(classification_report(all_labels, final_preds, zero_division=0))

    return {
        "threshold": best_t,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "confusion_matrix": cm
    }

In [23]:
best_val_f1 = -1.0
best_state = None

for epoch in range(1, 51):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_metrics = evaluate_model(model, val_loader, DEVICE, verbose=False)

    val_f1 = val_metrics["f1"]
    val_auc = val_metrics["auc"]

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val AUC: {val_auc:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_state, "coi_gpt_change_1.pt")

Epoch 001 | Train Loss: 1.1502 | Val AUC: 0.7721 | Val F1: 0.3870
Epoch 002 | Train Loss: 1.0329 | Val AUC: 0.8102 | Val F1: 0.4350
Epoch 003 | Train Loss: 0.9939 | Val AUC: 0.8176 | Val F1: 0.4477
Epoch 004 | Train Loss: 0.9660 | Val AUC: 0.8164 | Val F1: 0.4580
Epoch 005 | Train Loss: 0.9470 | Val AUC: 0.8272 | Val F1: 0.4688
Epoch 006 | Train Loss: 0.9290 | Val AUC: 0.8413 | Val F1: 0.4773
Epoch 007 | Train Loss: 0.9135 | Val AUC: 0.8398 | Val F1: 0.5028
Epoch 008 | Train Loss: 0.8979 | Val AUC: 0.8434 | Val F1: 0.5034
Epoch 009 | Train Loss: 0.8855 | Val AUC: 0.8454 | Val F1: 0.5198
Epoch 010 | Train Loss: 0.8831 | Val AUC: 0.8476 | Val F1: 0.5185
Epoch 011 | Train Loss: 0.8754 | Val AUC: 0.8452 | Val F1: 0.5135
Epoch 012 | Train Loss: 0.8686 | Val AUC: 0.8516 | Val F1: 0.5234
Epoch 013 | Train Loss: 0.8634 | Val AUC: 0.8511 | Val F1: 0.5205
Epoch 014 | Train Loss: 0.8567 | Val AUC: 0.8466 | Val F1: 0.5191
Epoch 015 | Train Loss: 0.8459 | Val AUC: 0.8556 | Val F1: 0.5278
Epoch 016 

KeyboardInterrupt: 

In [26]:
best_state = torch.load('coi_gpt_change_1.pt', map_location=DEVICE)
model.load_state_dict(best_state)

test_metrics = evaluate_model(model, test_loader, DEVICE, verbose=True)


===== EVALUATION =====
Best Threshold: 0.751
Accuracy:  0.9011
Precision: 0.5533
Recall:    0.5112
F1 Score:  0.5314
AUC:       0.8606

Confusion Matrix:
[[8285  444]
 [ 526  550]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.94      0.95      0.94      8729
         1.0       0.55      0.51      0.53      1076

    accuracy                           0.90      9805
   macro avg       0.75      0.73      0.74      9805
weighted avg       0.90      0.90      0.90      9805



## Other training method

In [10]:
@torch.no_grad()
def get_probs_and_labels(model, loader, device):
    model.eval()
    all_probs = []
    all_labels = []
    for xb, mb, yb in loader:
        xb, mb, yb = xb.to(device), mb.to(device), yb.to(device)
        logits, _ = model(xb, mask=mb)
        probs = torch.sigmoid(logits)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(yb.cpu().numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

def find_best_threshold(probs, labels, search_range=(0.1, 0.9, 0.01)):
    best_t = 0.5
    best_f1 = -1.0
    for t in np.arange(search_range[0], search_range[1] + search_range[2], search_range[2]):
        preds = (probs >= t).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    return best_t, best_f1

def evaluate_with_threshold(probs, labels, threshold):
    preds = (probs >= threshold).astype(int)
    return {
        "f1": f1_score(labels, preds, zero_division=0),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "accuracy": accuracy_score(labels, preds),
        "auc": roc_auc_score(labels, probs)
    }

In [5]:
@torch.no_grad()
def evaluate_model(model, loader, device, verbose=True):
    model.eval()
    all_probs, all_labels, all_masks = [], [], []
    
    for xb, mb, yb in loader:
        xb, mb = xb.to(device), mb.to(device)
        logits, _ = model(xb, mask=mb)
        probs = torch.sigmoid(logits)
        all_probs.append(probs.detach().cpu().numpy())
        all_labels.append(yb.detach().numpy())
        all_masks.append(mb.detach().cpu().numpy())
    
    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    masks = np.concatenate(all_masks)
    
    # Keep only samples with at least one valid time step AND non-NaN prob
    valid = masks.any(axis=1) & ~np.isnan(probs)
    if not valid.any():
        print("No valid samples – check your masks or model forward pass")
        return {"auc": 0.5, "f1": 0.0, "precision": 0.0, "recall": 0.0, "threshold": 0.5}
    
    probs = probs[valid]
    labels = labels[valid]
    
    # Find best threshold
    best_f1, best_t = 0, 0.5
    for t in np.linspace(0.1, 0.9, 81):
        preds = (probs >= t).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    
    preds = (probs >= best_t).astype(int)
    auc = roc_auc_score(labels, probs)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    
    if verbose:
        print(f"Threshold={best_t:.3f} | AUC={auc:.4f} | F1={best_f1:.4f} | P={precision:.3f} R={recall:.3f}")
    
    return {"threshold": best_t, "auc": auc, "f1": best_f1, "precision": precision, "recall": recall}

In [ ]:
best_val_auc = -1.0
best_val_f1 = -1.0
best_state = None

for epoch in range(1, 51):
    # Training
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    
    # Validation
    model.eval()
    val_loss = 0.0
    all_val_probs = []
    all_val_labels = []
    
    with torch.no_grad():
        for xb, mb, yb in val_loader:
            xb = xb.to(DEVICE)
            mb = mb.to(DEVICE)
            yb = yb.to(DEVICE)
            
            mb = mb.to(DEVICE).bool()   # add this line
            logits, _ = model(xb, mask=mb, return_attentions=False)
            
            # Calculate validation loss
            loss = criterion(logits, yb)
            val_loss += loss.item() * xb.size(0)
            
            # Get probabilities
            probs = torch.sigmoid(logits)
            all_val_probs.append(probs.cpu().numpy())
            all_val_labels.append(yb.cpu().numpy())
    
    # Aggregate metrics
    val_loss = val_loss / len(val_loader.dataset)
    val_probs = np.concatenate(all_val_probs)
    val_labels = np.concatenate(all_val_labels)
    # After collecting all_val_probs and all_val_labels, add this:

    # Remove NaN values (patients with no valid measurements)
    valid_mask = ~np.isnan(val_probs)
    if not valid_mask.all():
        print(f"Warning: Removing {np.sum(~valid_mask)} samples with NaN predictions")
        val_probs = val_probs[valid_mask]
        val_labels = val_labels[valid_mask]

    # Also check for any remaining NaN
    if np.any(np.isnan(val_probs)):
        print("ERROR: Still have NaN values!")
        # Replace with 0.5 (neutral prediction) as fallback
        val_probs = np.nan_to_num(val_probs, nan=0.5)
        
    # Calculate AUC and F1 with optimal threshold
    val_auc = roc_auc_score(val_labels, val_probs)
    
    # Find best threshold for F1 on validation set
    best_thresh = 0.5
    best_f1 = 0.0
    for thresh in np.linspace(0.1, 0.9, 81):
        preds = (val_probs >= thresh).astype(int)
        f1 = f1_score(val_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    
    val_f1 = best_f1
    
    # Save best model based on AUC (or change to F1 if preferred)
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_val_f1 = val_f1  # Store corresponding F1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_state, "best_model_eicu.pt")
        print(f"Epoch {epoch}: ✓ New best model (AUC: {val_auc:.4f}, F1: {val_f1:.4f})")
    
    # Print all metrics
    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f} (threshold={best_thresh:.2f})")

print(f"\nTraining complete. Best validation AUC: {best_val_auc:.4f}, Best validation F1: {best_val_f1:.4f}")

Epoch 1: ✓ New best model (AUC: 0.8166, F1: 0.4143)
Epoch 001 | Train Loss: 1.1257 | Val Loss: 1.0244 | Val AUC: 0.8166 | Val F1: 0.4143 (threshold=0.68)
Epoch 2: ✓ New best model (AUC: 0.8354, F1: 0.4474)
Epoch 002 | Train Loss: 1.0001 | Val Loss: 0.9320 | Val AUC: 0.8354 | Val F1: 0.4474 (threshold=0.71)
Epoch 3: ✓ New best model (AUC: 0.8393, F1: 0.4659)
Epoch 003 | Train Loss: 0.9556 | Val Loss: 0.9089 | Val AUC: 0.8393 | Val F1: 0.4659 (threshold=0.76)
Epoch 4: ✓ New best model (AUC: 0.8460, F1: 0.4784)
Epoch 004 | Train Loss: 0.9295 | Val Loss: 0.8855 | Val AUC: 0.8460 | Val F1: 0.4784 (threshold=0.71)
Epoch 5: ✓ New best model (AUC: 0.8495, F1: 0.4857)
Epoch 005 | Train Loss: 0.9154 | Val Loss: 0.9148 | Val AUC: 0.8495 | Val F1: 0.4857 (threshold=0.67)
Epoch 6: ✓ New best model (AUC: 0.8556, F1: 0.4884)
Epoch 006 | Train Loss: 0.8988 | Val Loss: 0.8841 | Val AUC: 0.8556 | Val F1: 0.4884 (threshold=0.71)
Epoch 7: ✓ New best model (AUC: 0.8575, F1: 0.4889)
Epoch 007 | Train Loss: 

KeyboardInterrupt: 

In [6]:
from imblearn.over_sampling import SMOTE

# ── TSMOTE for temporal data ──────────────────────────────────────────────────
# SMOTE can't handle 3D directly — flatten, resample, reshape

N, T, F = X_train.shape
X_flat = X_train.reshape(N, T * F)  # (N, T*F)

target_positive  = int(0.30 * len(y_train) / 0.70)  # how many positives needed for 70/30
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
sampling_strategy = target_positive / n_neg  # ratio of minority to majority

smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42, k_neighbors=5)
X_res, y_res = smote.fit_resample(X_flat, y_train)

# reshape back to 3D
X_train_sm = X_res.reshape(-1, T, F)
y_train_sm = y_res

print(f"Before SMOTE: {N} samples, {y_train.mean()*100:.1f}% positive")
print(f"After  SMOTE: {len(y_train_sm)} samples, {y_train_sm.mean()*100:.1f}% positive")

# ── rebuild mask for synthetic samples ────────────────────────────────────────
# synthetic samples don't have a real mask — use all-ones (fully observed)
N_new = len(X_train_sm)
mask_train_sm = np.ones((N_new, T), dtype=np.float32)
# preserve original masks for real samples
mask_train_sm[:N] = mask_train  

# ── dataloaders ───────────────────────────────────────────────────────────────
train_ds = ICUDataset(X_train_sm, mask_train_sm, y_train_sm)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

Before SMOTE: 45756 samples, 11.0% positive
After  SMOTE: 60343 samples, 32.5% positive


In [7]:
import torch
import torch.nn as nn

# ── RETAIN Model ─────────────────────────────────────────────────────────────

class RETAINModel(nn.Module):
    """
    RETAIN: Reverse Time Attention Model (Choi et al., 2016)
    - Alpha: visit-level attention (how important is each timestep)
    - Beta:  variable-level attention (how important is each feature at each timestep)
    """
    def __init__(self, input_dim, embed_dim=128, dropout=0.3):
        super().__init__()
        self.input_dim = input_dim
        self.embed_dim = embed_dim

        # input embedding
        self.embedding = nn.Linear(input_dim, embed_dim)

        # RNN for alpha (visit attention) — runs in reverse
        self.rnn_alpha = nn.GRU(embed_dim, embed_dim, batch_first=True)

        # RNN for beta (variable attention) — runs in reverse
        self.rnn_beta  = nn.GRU(embed_dim, embed_dim, batch_first=True)

        # attention projections
        self.attn_alpha = nn.Linear(embed_dim, 1)          # → scalar per timestep
        self.attn_beta  = nn.Linear(embed_dim, input_dim)  # → vector per feature

        self.dropout = nn.Dropout(dropout)
        self.output  = nn.Linear(input_dim, 1)

    def forward(self, x, mask=None, return_attentions=False):
        """
        x    : (B, T, F)
        mask : (B, T) — 1 = valid, 0 = pad  [optional]
        returns (logits (B,), attentions dict or None)
        """
        B, T, F = x.shape

        # embed each timestep
        emb = torch.relu(self.embedding(x))   # (B, T, E)
        emb = self.dropout(emb)

        # reverse the sequence for both RNNs
        emb_rev = torch.flip(emb, dims=[1])   # (B, T, E)

        # alpha RNN → visit-level attention
        h_alpha, _ = self.rnn_alpha(emb_rev)  # (B, T, E)
        h_alpha    = self.dropout(h_alpha)
        e_alpha    = self.attn_alpha(h_alpha).squeeze(-1)  # (B, T)

        # mask padding before softmax
        if mask is not None:
            mask_rev = torch.flip(mask, dims=[1]).float()  # (B, T)
            e_alpha  = e_alpha.masked_fill(mask_rev == 0, -1e9)

        alpha = torch.softmax(e_alpha, dim=1)  # (B, T)

        # beta RNN → variable-level attention
        h_beta, _ = self.rnn_beta(emb_rev)    # (B, T, E)
        h_beta     = self.dropout(h_beta)
        beta       = torch.tanh(self.attn_beta(h_beta))  # (B, T, F)

        # flip both attention outputs back to forward order
        alpha = torch.flip(alpha, dims=[1])    # (B, T)
        beta  = torch.flip(beta,  dims=[1])    # (B, T, F)

        # context vector: weighted sum over timesteps
        # alpha: (B, T) → (B, T, 1), beta: (B, T, F), x: (B, T, F)
        context = (alpha.unsqueeze(-1) * beta * x).sum(dim=1)  # (B, F)
        context = self.dropout(context)

        logits = self.output(context).squeeze(-1)  # (B,)

        if return_attentions:
            return logits, {"alpha": alpha, "beta": beta}
        return logits, None


# ── Build components ──────────────────────────────────────────────────────────

def build_retain_components(X_train, y_train, device,
                            embed_dim=128, dropout=0.3, lr=1e-3):
    input_dim = X_train.shape[2]  # F

    model = RETAINModel(input_dim=input_dim, embed_dim=embed_dim,
                        dropout=dropout).to(device)

    n_neg      = (y_train == 0).sum()
    n_pos      = (y_train == 1).sum()
    pos_weight = torch.tensor([(n_neg / n_pos) * 0.6],
                               dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss()

    optimizer  = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    total = sum(p.numel() for p in model.parameters())
    print(f"RETAIN | input_dim={input_dim} | embed_dim={embed_dim}")
    print(f"Total parameters: {total:,}")
    print(f"pos_weight: {pos_weight.item():.2f}")

    return model, criterion, optimizer


# ── Training loop (drop-in, same as CoI) ─────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, device, grad_clip=1.0):
    model.train()
    total_loss = 0.0

    for xb, mb, yb in loader:
        xb = xb.to(device)
        mb = mb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(xb, mask=mb, return_attentions=False)
        loss = criterion(logits, yb)
        loss.backward()

        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset)


# ── Run ───────────────────────────────────────────────────────────────────────

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model, criterion, optimizer = build_retain_components(
    X_train, y_train, DEVICE,
    embed_dim=128,
    dropout=0.3,
    lr=1e-3
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=5, verbose=True
)

best_auc   = 0.0
best_state = None
patience_counter = 0
EARLY_STOP = 15

for epoch in range(1, 101):
    train_loss   = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_metrics  = evaluate_model(model, val_loader, DEVICE, verbose=False)

    val_auc = val_metrics["auc"]
    val_f1  = val_metrics["f1"]

    scheduler.step(val_auc)

    if val_auc > best_auc:
        best_auc   = val_auc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        tag = "✓ best"
    else:
        patience_counter += 1
        tag = f"patience {patience_counter}/{EARLY_STOP}"

    print(f"Epoch {epoch:03d} | Loss: {train_loss:.4f} | "
          f"Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f} | {tag}")

    if patience_counter >= EARLY_STOP:
        print(f"Early stopping at epoch {epoch}")
        break

# restore best
model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
print(f"\nBest Val AUC: {best_auc:.4f}")

RETAIN | input_dim=30 | embed_dim=128
Total parameters: 206,142
pos_weight: 4.87
Epoch 001 | Loss: 0.4405 | Val AUC: 0.8282 | Val F1: 0.4769 | ✓ best
Epoch 002 | Loss: 0.3595 | Val AUC: 0.8542 | Val F1: 0.5144 | ✓ best
Epoch 003 | Loss: 0.3308 | Val AUC: 0.8593 | Val F1: 0.5160 | ✓ best
Epoch 004 | Loss: 0.3110 | Val AUC: 0.8539 | Val F1: 0.4940 | patience 1/15
Epoch 005 | Loss: 0.2942 | Val AUC: 0.8595 | Val F1: 0.5120 | ✓ best
Epoch 006 | Loss: 0.2868 | Val AUC: 0.8637 | Val F1: 0.5314 | ✓ best
Epoch 007 | Loss: 0.2770 | Val AUC: 0.8590 | Val F1: 0.5297 | patience 1/15
Epoch 008 | Loss: 0.2695 | Val AUC: 0.8651 | Val F1: 0.5273 | ✓ best
Epoch 009 | Loss: 0.2639 | Val AUC: 0.8620 | Val F1: 0.5186 | patience 1/15
Epoch 010 | Loss: 0.2568 | Val AUC: 0.8572 | Val F1: 0.5124 | patience 2/15
Epoch 011 | Loss: 0.2521 | Val AUC: 0.8520 | Val F1: 0.4950 | patience 3/15
Epoch 012 | Loss: 0.2490 | Val AUC: 0.8661 | Val F1: 0.5269 | ✓ best
Epoch 013 | Loss: 0.2433 | Val AUC: 0.8674 | Val F1: 0.5

In [30]:
# ── Step 1: find best threshold on val set ────────────────────────────────────
model.eval()
all_probs, all_labels = [], []

with torch.no_grad():
    for xb, mb, yb in val_loader:
        logits, _ = model(xb.to(DEVICE), mask=mb.to(DEVICE))
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(yb.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

thresholds = np.arange(0.1, 0.9, 0.01)
f1s = [f1_score(all_labels, all_probs >= t) for t in thresholds]
best_t  = thresholds[np.argmax(f1s)]
best_f1 = max(f1s)
print(f"Best threshold: {best_t:.2f} → Val F1: {best_f1:.4f}")

# ── Step 2: evaluate on test set using that threshold ─────────────────────────
all_probs_test, all_labels_test = [], []

with torch.no_grad():
    for xb, mb, yb in test_loader:
        logits, _ = model(xb.to(DEVICE), mask=mb.to(DEVICE))
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs_test.extend(probs)
        all_labels_test.extend(yb.numpy())

all_probs_test  = np.array(all_probs_test)
all_labels_test = np.array(all_labels_test)

preds = (all_probs_test >= best_t).astype(int)

from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

print(f"\n=== Test Evaluation (threshold={best_t:.2f}) ===")
print(f"AUC:  {roc_auc_score(all_labels_test, all_probs_test):.4f}")
print(f"F1:   {f1_score(all_labels_test, preds):.4f}")
print(confusion_matrix(all_labels_test, preds))
print(classification_report(all_labels_test, preds))

Best threshold: 0.69 → Val F1: 0.5645

=== Test Evaluation (threshold=0.69) ===
AUC:  0.8871
F1:   0.5588
[[8304  425]
 [ 494  582]]
              precision    recall  f1-score   support

         0.0       0.94      0.95      0.95      8729
         1.0       0.58      0.54      0.56      1076

    accuracy                           0.91      9805
   macro avg       0.76      0.75      0.75      9805
weighted avg       0.90      0.91      0.90      9805

